# Helper Notebook for Dataset Splitting
This notebook is to help with splitting the images and labels across for Google Street View (GSV) to `test` and actual parking sign photographs to `val`.  
All the photographs and labels are originally placed in the `train` folder then get moved accordingly, ensuring an approximate 70-30 split for all classes.  
This is for Data Science Capstone project titled **_Australian Parking Sign Detection and Structured Information Extraction_**.  

By Harry Ngo

In [ ]:
import os
import shutil
import re
import random
from collections import defaultdict

Moving GSV files to `test` folders.

In [ ]:
# paths
images_dir = "dataset/images/train"
labels_dir = "dataset/labels/train"
test_images_dir = "dataset/images/test"
test_labels_dir = "dataset/labels/test"

# create test directories
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_labels_dir, exist_ok=True)

# regex for Google Street View low res images
pattern = re.compile(r"^[A-Za-z]+_IMG_\d+\.png$")

for img_name in os.listdir(images_dir):
    if pattern.match(img_name):
        img_path = os.path.join(images_dir, img_name)
        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(labels_dir, label_name)

        # move image
        shutil.move(img_path, os.path.join(test_images_dir, img_name))

        # move label if it exists
        if os.path.exists(label_path):
            shutil.move(label_path, os.path.join(test_labels_dir, label_name))

print("Moved low-resolution GSV images and labels to test/")

Moving photographs to `val` folders and performing stratified splitting.

In [ ]:
# base paths
base = "dataset"
dirs = {
    "train_img": os.path.join(base, "images/train"),
    "val_img": os.path.join(base, "images/val"),
    "train_lbl": os.path.join(base, "labels/train"),
    "val_lbl": os.path.join(base, "labels/val"),
}

os.makedirs(dirs["val_img"], exist_ok=True)
os.makedirs(dirs["val_lbl"], exist_ok=True)

# load labels
image_classes = defaultdict(set)
label_files = [f for f in os.listdir(dirs["train_lbl"]) if f.endswith(".txt")]
for lbl in label_files:
    with open(os.path.join(dirs["train_lbl"], lbl)) as f:
        for line in f:
            cls = int(line.split()[0])
            image_classes[lbl].add(cls)

# group images
G1 = []   # only class 1
G2 = []   # only class 2
G3 = []   # both 1 & 2
Gother = []  # everything else

for lbl, cls in image_classes.items():
    if cls == {1}:
        G1.append(lbl)
    elif cls == {2}:
        G2.append(lbl)
    elif {1, 2}.issubset(cls):
        G3.append(lbl)
    else:
        Gother.append(lbl)

random.seed(42)
random.shuffle(G1)
random.shuffle(G2)
random.shuffle(G3)
random.shuffle(Gother)

def split(group):
    s = int(0.7 * len(group))
    return group[:s], group[s:]

# independent stratification
train_G1, val_G1 = split(G1)
train_G2, val_G2 = split(G2)

# mixed class handling
need_val_c1 = max(0, int(0.3 * len(G1)) - len(val_G1))
need_val_c2 = max(0, int(0.3 * len(G2)) - len(val_G2))
val_G3 = G3[:max(need_val_c1, need_val_c2)]
train_G3 = G3[max(need_val_c1, need_val_c2):]

# other classes get random split
train_Gother, val_Gother = split(Gother)

train_subset = train_G1 + train_G2 + train_G3 + train_Gother
val_subset = val_G1 + val_G2 + val_G3 + val_Gother

def move(lbl_list, dest_img, dest_lbl):
    for lbl in lbl_list:
        root = os.path.splitext(lbl)[0]
        img_file = None
        for ext in (".jpg", ".jpeg", ".png"):
            cand = root + ext
            if os.path.exists(os.path.join(dirs["train_img"], cand)):
                img_file = cand
                break
        if img_file is None:
            print("Missing image for:", lbl)
            continue
        shutil.move(os.path.join(dirs["train_img"], img_file),
                    os.path.join(dest_img, img_file))
        shutil.move(os.path.join(dirs["train_lbl"], lbl),
                    os.path.join(dest_lbl, lbl))

move(val_subset, dirs["val_img"], dirs["val_lbl"])
print("Split completed")
print(f"Train images: {len(os.listdir(dirs['train_img']))}")
print(f"Val images: {len(os.listdir(dirs['val_img']))}")

In [ ]:
def class_distribution(lbl_dir):
    counts = defaultdict(int)
    for f in os.listdir(lbl_dir):
        full_path = os.path.join(lbl_dir, f)
        if not os.path.isfile(full_path) or f.startswith(".") or not f.endswith(".txt"):
            continue
        with open(full_path) as lf:
            for line in lf:
                cls = int(line.split()[0])
                counts[cls] += 1
    return dict(counts)

print("TRAIN:", class_distribution(dirs["train_lbl"]))
print("VAL:", class_distribution(dirs["val_lbl"]))

In [ ]:
# move back some class 2 samples from val to train
MOVE_COUNT = 3

class2_labels = []
for lbl in os.listdir(dirs["val_lbl"]):
    if not lbl.endswith(".txt"):
        continue
    with open(os.path.join(dirs["val_lbl"], lbl)) as f:
        for line in f:
            if line.startswith("2 "):
                class2_labels.append(lbl)
                break

print("Class 2 in val:", len(class2_labels))

to_move = class2_labels[:MOVE_COUNT]
for lbl in to_move:
    root = os.path.splitext(lbl)[0]
    img_file = None
    for ext in (".jpg", ".jpeg", ".png"):
        candidate = root + ext
        if os.path.exists(os.path.join(dirs["val_img"], candidate)):
            img_file = candidate
            break
    if img_file is None:
        print("Missing image for", lbl)
        continue
    print(f"Moving back: {lbl} / {img_file}")
    shutil.move(os.path.join(dirs["val_lbl"], lbl),
                os.path.join(dirs["train_lbl"], lbl))
    shutil.move(os.path.join(dirs["val_img"], img_file),
                os.path.join(dirs["train_img"], img_file))

print("TRAIN:", class_distribution(dirs["train_lbl"]))
print("VAL:", class_distribution(dirs["val_lbl"]))

| Class | Train | Val | Total | % Train |   % Val | 
| ----- | ----: | --: | ----: | ------: | ------: | 
| **0: parking-sign** |   442 | 160 |   602 |     73% |     27% | 
| **1: no-parking-symbol** |    50 |  18 |    68 |     74% |     26% | 
| **2: disabled-symbol** |    16 |   7 |    23 | 70% | 30% | 
| **3: text-region-full** |   420 | 151 |   571 |     74% |     26% | 